In [66]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [67]:
# 폰트 설정
import matplotlib as mpl
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = "retina"
!apt install fonts-nanum

import matplotlib.font_manager as fm
fontpath = "/usr/share/fonts/truetype/nanum/NanumMyeongjo.ttf"
font = fm.FontProperties(fname = fontpath, size = 9)

import matplotlib as mpl
mpl.font_manager._rebuild()
mpl.pyplot.rc("font", family = "NanumMyeongjo")

# 이 셀 실행하고 런타임 - 런타임 다시 시작 하고 난 다음 다시 이 셀 실행

Reading package lists... Done
Building dependency tree       
Reading state information... Done
fonts-nanum is already the newest version (20170925-1).
The following package was automatically installed and is no longer required:
  libnvidia-common-460
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 20 not upgraded.


In [68]:
# ebm 쓰기 위해 interpret 설치
! pip install interpret

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [69]:
 # Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# warning
import warnings
warnings.filterwarnings('ignore')

# model
import xgboost
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show

# 봄

In [70]:
data_봄 = pd.read_csv("/content/gdrive/MyDrive/산단공_프로젝트/봄_중업종재해반영x.csv", encoding = "cp949")

In [71]:
id_봄 = pd.Series(data_봄.id, name = 'id') # 그리드 id만 추출해서 저장
data_봄.drop(['Unnamed: 0', 'Unnamed: 0.1', "단지명", "id"], axis = 1, inplace = True)

In [72]:
# 중대재해가 1번 이상 발생했으면 1, 발생하지 않았으면 0

data_봄.loc[data_봄['중대재해유무'] == True, "중대재해유무"] = 1
data_봄.loc[data_봄['중대재해유무'] == False, "중대재해유무"] = 0

# 필요없는 컬럼 제거
data_봄.drop(['발생형태_끼임', '발생형태_작업관련질병(뇌심등)',
       '발생형태_직업병(진폐제외)', '발생형태_넘어짐', '발생형태_물체에맞음', '발생형태_부딪힘', '발생형태_떨어짐',
       '발생형태_절단·베임·찔림', '발생형태_불균형및무리한동작', '발생형태_이상온도접촉', '발생형태_깔림·뒤집힘',
       '발생형태_사업장외교통사고', '발생형태_체육행사등의사고', '발생형태_화학물질누출·접촉', '발생형태_폭발·파열',
       '발생형태_화재', '발생형태_감전', '발생형태_진폐', '발생형태_폭력행위', '발생형태_무너짐',
       '발생형태_사업장내교통사고', '발생형태_기타', '발생형태_분류불능', '발생형태_산소결핍', '발생형태_빠짐ㆍ익사',
       '발생형태_동물상해', '근속기간_10년이상', '근속기간_1년미만', '근속기간_5년~10년미만', '근속기간_1년~2년미만',
       '근속기간_2년~3년미만', '근속기간_3년~4년미만', '근속기간_4년~5년미만', '연령_50세~59세',
       '연령_60세~69세', '연령_40세~49세', '연령_30세~39세', '연령_20세~29세', '연령_70세이상', ], axis = 1, inplace = True)

In [73]:
# 관리기관 원-핫 인코딩
data_봄 = pd.get_dummies(data = data_봄, columns = ['관리기관'], prefix = '관리기관')

In [74]:
# 변수 스케일링

from sklearn.preprocessing import MinMaxScaler
scaler_ = MinMaxScaler()
data_봄_scaled_ = scaler_.fit_transform(data_봄)

data_봄 = pd.DataFrame(data_봄_scaled_, columns = data_봄.columns)

In [75]:
data_봄

,근로자수,근로손실일,중대재해유무,그리드 내 총 기업 수,매출액,부채비율,"중업종분류_기계기구,금속,비금속광물제품제조업","중업종분류_전기기계기구,정밀기구,전자제품제조업","중업종분류_전문,보건,교욱,여가관련서비스","중업종분류_출판,인쇄,제본또는인쇄물가공업",...,관리기관_한국산업단지공단 경북지역본부,관리기관_한국산업단지공단 대구지역본부 달성사무소,관리기관_한국산업단지공단 부산지역본부,관리기관_한국산업단지공단 부산지역본부 사하사무소,관리기관_한국산업단지공단 서울지역본부,관리기관_한국산업단지공단 시화지사 시화MTV사무소,관리기관_한국산업단지공단 울산지역본부,관리기관_한국산업단지공단 인천지역본부,관리기관_한국산업단지공단 전남지역본부 대불지사,관리기관_한국산업단지공단 전북지역본부
0,1.000000,0.040611,1.0,0.005222,0.146193,0.168453,0.006623,0.003704,0.017094,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.303626,0.037477,1.0,0.003916,0.180756,0.168682,0.003311,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.209815,0.036543,1.0,0.000000,0.139289,0.167667,0.003311,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.142747,0.034142,1.0,0.003916,0.414165,0.168950,0.003311,0.000000,0.008547,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.024503,0.010936,1.0,0.000000,0.022083,0.168333,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,0.001470,0.005468,0.0,0.002611,0.000683,0.167739,0.000000,0.011111,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1613,0.000700,0.023340,0.0,0.002611,0.000206,0.170686,0.000000,0.000000,0.000000,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1614,0.001540,0.006402,0.0,0.007833,0.000178,0.170065,0.016556,0.003704,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1615,0.004201,0.016404,0.0,0.007833,0.000924,0.168125,0.023179,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [76]:
# 중대재해유무: target

X_봄 = data_봄.drop(["중대재해유무"], axis = 1)
y_봄 = data_봄.중대재해유무

In [77]:
# 학습 데이터와 테스트 데이터 나눔
# 중대재해발생유무 비율에 맞도록 분리

X_train_봄, X_test_봄, y_train_봄, y_test_봄 = train_test_split(X_봄, y_봄, random_state = 42, stratify=y_봄)

In [78]:
# 평가지표 함수

def get_clf_eval(y_test, y_pred):
    confusion = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    F1 = f1_score(y_test, y_pred)
    AUC = roc_auc_score(y_test, y_pred)
    print('오차행렬:\n', confusion)
    print('\n정확도: {:.4f}'.format(accuracy))
    print('정밀도: {:.4f}'.format(precision))
    print('재현율: {:.4f}'.format(recall))
    print('F1: {:.4f}'.format(F1))
    print('AUC: {:.4f}'.format(AUC))

## RandomForest

In [79]:
rf_clf_봄 = RandomForestClassifier()
rf_clf_봄.fit(X_train_봄, y_train_봄)

y_rf_pred_봄 = rf_clf_봄.predict(X_test_봄)

get_clf_eval(y_test_봄, y_rf_pred_봄)

오차행렬:
 [[199  37]
 [ 25 144]]

정확도: 0.8469
정밀도: 0.7956
재현율: 0.8521
F1: 0.8229
AUC: 0.8476


In [80]:
# RF 모델 하이퍼 파라미터 튜닝

rf_봄_best = RandomForestClassifier()

# 파라미터 지정
rf_봄_parameters ={ 'n_estimators' : [100, 200, 300],
           'max_depth' : [6, 8, 10, 12],
           'min_samples_leaf' : [1, 3, 5, 8],
           'min_samples_split' : [1, 3, 5, 8]
            }
rf_grid_봄=GridSearchCV(rf_봄_best, param_grid = rf_봄_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


rf_grid_봄.fit(X_train_봄, y_train_봄)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(rf_grid_봄.best_score_))
print("best param : ",rf_grid_봄.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(rf_grid_봄.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.8735
best param :  {'max_depth': 12, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}


,params,mean_test_score,rank_test_score
152,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.873509,1
164,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.871560,2
103,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.871548,3
107,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.871548,3
59,"{'max_depth': 8, 'min_samples_leaf': 1, 'min_s...",0.871548,3
104,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.871548,3
151,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.871548,3
155,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.871537,8
105,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.871537,8
154,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.871525,10


In [81]:
estimator_rf_봄 = rf_grid_봄.best_estimator_
y_best_pred_봄_rf = estimator_rf_봄.predict(X_test_봄)
get_clf_eval(y_test_봄, y_best_pred_봄_rf)

오차행렬:
 [[198  38]
 [ 21 148]]

정확도: 0.8543
정밀도: 0.7957
재현율: 0.8757
F1: 0.8338
AUC: 0.8574


## 서포트 벡터

In [82]:
svc_봄 = SVC(kernel='poly').fit(X_train_봄, y_train_봄)
svc_봄.fit(X_train_봄, y_train_봄)

y_svc_pred_봄 = svc_봄.predict(X_test_봄)

get_clf_eval(y_test_봄, y_svc_pred_봄)

오차행렬:
 [[204  32]
 [121  48]]

정확도: 0.6222
정밀도: 0.6000
재현율: 0.2840
F1: 0.3855
AUC: 0.5742


## XGB

In [83]:
xgb_봄 = XGBClassifier(early_sropping_rounds = 50, random_state = 42)
xgb_봄.fit(X_train_봄, y_train_봄)

y_xgb_pred_봄 = xgb_봄.predict(X_test_봄)

get_clf_eval(y_test_봄, y_xgb_pred_봄)

오차행렬:
 [[197  39]
 [ 27 142]]

정확도: 0.8370
정밀도: 0.7845
재현율: 0.8402
F1: 0.8114
AUC: 0.8375


In [84]:
# XGB 모델 하이퍼 파라미터 튜닝

xgb_봄_best = XGBClassifier(early_sropping_rounds = 50, random_state = 42)

# 파라미터 지정
xg_봄_parameters ={'n_estimators' : [100,200,300],
    'learning_rate' : [0.01,0.05,0.1,0.15],
    'max_depth' : [3,5,7,10],
    'gamma' : [0,1,2,3],
    'colsample_bytree' : [0.8,0.9]}
xgb_grid_봄=GridSearchCV(xgb_봄_best, param_grid = xg_봄_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


xgb_grid_봄.fit(X_train_봄, y_train_봄)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(xgb_grid_봄.best_score_))
print("best param : ",xgb_grid_봄.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(xgb_grid_봄.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.8696
best param :  {'colsample_bytree': 0.8, 'gamma': 3, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}


,params,mean_test_score,rank_test_score
156,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.869576,1
158,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.869576,1
157,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.867604,3
338,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.863671,4
349,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.863659,5
348,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.863659,5
350,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.863659,5
360,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.861675,8
361,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.861675,8
146,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.861675,8


In [85]:
estimator_xgb_봄 = xgb_grid_봄.best_estimator_
y_best_pred_봄_xgb = estimator_xgb_봄.predict(X_test_봄)
get_clf_eval(y_test_봄, y_best_pred_봄_xgb)

오차행렬:
 [[196  40]
 [ 22 147]]

정확도: 0.8469
정밀도: 0.7861
재현율: 0.8698
F1: 0.8258
AUC: 0.8502


In [132]:
# 재현율이 가장 높은 RF모델 저장
import joblib

joblib.dump(estimator_rf_봄 ,"/content/gdrive/MyDrive/프로젝트_모델/rf_봄_최종.pkl")

['/content/gdrive/MyDrive/프로젝트_모델/rf_봄_최종.pkl']

# 여름

In [87]:
data_여름 = pd.read_csv("/content/gdrive/MyDrive/산단공_프로젝트/여름_중업종재해반영x.csv", encoding = "cp949")

In [88]:
id_여름 = pd.Series(data_여름.id, name = 'id') # 그리드 id만 추출해서 저장
data_여름.drop(['Unnamed: 0', 'Unnamed: 0.1', "단지명", "id"], axis = 1, inplace = True)

In [89]:
# 중대재해가 1번 이상 발생했으면 1, 발생하지 않았으면 0

data_여름.loc[data_여름['중대재해유무'] == True, "중대재해유무"] = 1
data_여름.loc[data_여름['중대재해유무'] == False, "중대재해유무"] = 0

# 필요없는 컬럼 제거
data_여름.drop(['발생형태_끼임', '발생형태_작업관련질병(뇌심등)',
       '발생형태_직업병(진폐제외)', '발생형태_넘어짐', '발생형태_물체에맞음', '발생형태_부딪힘', '발생형태_떨어짐',
       '발생형태_절단·베임·찔림', '발생형태_불균형및무리한동작', '발생형태_이상온도접촉', '발생형태_깔림·뒤집힘',
       '발생형태_사업장외교통사고', '발생형태_체육행사등의사고', '발생형태_화학물질누출·접촉', '발생형태_폭발·파열',
       '발생형태_화재', '발생형태_감전', '발생형태_진폐', '발생형태_폭력행위', '발생형태_무너짐',
       '발생형태_사업장내교통사고', '발생형태_기타', '발생형태_분류불능', '발생형태_산소결핍', '발생형태_빠짐ㆍ익사',
       '발생형태_동물상해', '근속기간_10년이상', '근속기간_1년미만', '근속기간_5년~10년미만', '근속기간_1년~2년미만',
       '근속기간_2년~3년미만', '근속기간_3년~4년미만', '근속기간_4년~5년미만', '연령_50세~59세',
       '연령_60세~69세', '연령_40세~49세', '연령_30세~39세', '연령_20세~29세', '연령_70세이상', ], axis = 1, inplace = True)

In [90]:
# 관리기관 원-핫 인코딩
data_여름 = pd.get_dummies(data = data_여름, columns = ['관리기관'], prefix = '관리기관')

In [91]:
# 변수 스케일링

from sklearn.preprocessing import MinMaxScaler
scaler_ = MinMaxScaler()
data_여름_scaled_ = scaler_.fit_transform(data_여름)

data_여름 = pd.DataFrame(data_여름_scaled_, columns = data_여름.columns)

In [92]:
# 중대재해유무: target

X_여름 = data_여름.drop(["중대재해유무"], axis = 1)
y_여름 = data_여름.중대재해유무

In [93]:
# 학습 데이터와 테스트 데이터 나눔
# 중대재해발생유무 비율에 맞도록 분리

X_train_여름, X_test_여름, y_train_여름, y_test_여름 = train_test_split(X_여름, y_여름, random_state = 42, stratify=y_여름)

## RandomForest

In [94]:
rf_clf_여름 = RandomForestClassifier()
rf_clf_여름.fit(X_train_여름, y_train_여름)

y_rf_pred_여름 = rf_clf_여름.predict(X_test_여름)

get_clf_eval(y_test_여름, y_rf_pred_여름)

오차행렬:
 [[242  19]
 [ 30 126]]

정확도: 0.8825
정밀도: 0.8690
재현율: 0.8077
F1: 0.8372
AUC: 0.8674


In [95]:
# RF 모델 하이퍼 파라미터 튜닝

rf_여름_best = RandomForestClassifier()

# 파라미터 지정
rf_여름_parameters ={ 'n_estimators' : [100, 200, 300],
           'max_depth' : [6, 8, 10, 12],
           'min_samples_leaf' : [1, 3, 5, 8],
           'min_samples_split' : [1, 3, 5, 8]
            }
rf_grid_여름=GridSearchCV(rf_여름_best, param_grid = rf_여름_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


rf_grid_여름.fit(X_train_여름, y_train_여름)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(rf_grid_여름.best_score_))
print("best param : ",rf_grid_여름.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(rf_grid_여름.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.7350
best param :  {'max_depth': 12, 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 100}


,params,mean_test_score,rank_test_score
153,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.735043,1
155,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.730769,2
106,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.730769,2
105,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.730769,2
166,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.730769,2
129,"{'max_depth': 10, 'min_samples_leaf': 5, 'min_...",0.730769,2
75,"{'max_depth': 8, 'min_samples_leaf': 5, 'min_s...",0.730769,2
152,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.728632,8
57,"{'max_depth': 8, 'min_samples_leaf': 1, 'min_s...",0.728632,8
137,"{'max_depth': 10, 'min_samples_leaf': 8, 'min_...",0.728632,8


In [96]:
estimator_rf_여름 = rf_grid_여름.best_estimator_
y_best_pred_여름_rf = estimator_rf_여름.predict(X_test_여름)
get_clf_eval(y_test_여름, y_best_pred_여름_rf)

오차행렬:
 [[240  21]
 [ 29 127]]

정확도: 0.8801
정밀도: 0.8581
재현율: 0.8141
F1: 0.8355
AUC: 0.8668


## 서포트 벡터

In [97]:
svc_여름 = SVC(kernel='poly').fit(X_train_여름, y_train_여름)
svc_여름.fit(X_train_여름, y_train_여름)

y_svc_pred_여름 = svc_여름.predict(X_test_여름)

get_clf_eval(y_test_여름, y_svc_pred_여름)

오차행렬:
 [[240  21]
 [118  38]]

정확도: 0.6667
정밀도: 0.6441
재현율: 0.2436
F1: 0.3535
AUC: 0.5816


## XGB

In [98]:
xgb_여름 = XGBClassifier(early_sropping_rounds = 50, random_state = 42)
xgb_여름.fit(X_train_여름, y_train_여름)

y_xgb_pred_여름 = xgb_여름.predict(X_test_여름)

get_clf_eval(y_test_여름, y_xgb_pred_여름)

오차행렬:
 [[242  19]
 [ 29 127]]

정확도: 0.8849
정밀도: 0.8699
재현율: 0.8141
F1: 0.8411
AUC: 0.8707


In [99]:
# XGB 모델 하이퍼 파라미터 튜닝

xgb_여름_best = XGBClassifier(early_sropping_rounds = 50, random_state = 42)

# 파라미터 지정
xg_여름_parameters ={'n_estimators' : [100,200,300],
    'learning_rate' : [0.01,0.05,0.1,0.15],
    'max_depth' : [3,5,7,10],
    'gamma' : [0,1,2,3],
    'colsample_bytree' : [0.8,0.9]}
xgb_grid_여름=GridSearchCV(xgb_여름_best, param_grid = xg_여름_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


xgb_grid_여름.fit(X_train_여름, y_train_여름)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(xgb_grid_여름.best_score_))
print("best param : ",xgb_grid_여름.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(xgb_grid_여름.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.7457
best param :  {'colsample_bytree': 0.9, 'gamma': 3, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}


,params,mean_test_score,rank_test_score
381,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.745726,1
371,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.745726,1
365,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.745726,1
370,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.743590,4
190,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.743590,4
382,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.743590,4
128,"{'colsample_bytree': 0.8, 'gamma': 2, 'learnin...",0.743590,4
257,"{'colsample_bytree': 0.9, 'gamma': 1, 'learnin...",0.743590,4
191,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.741453,9
176,"{'colsample_bytree': 0.8, 'gamma': 3, 'learnin...",0.741453,9


In [100]:
estimator_xgb_여름 = xgb_grid_여름.best_estimator_
y_best_pred_여름_xgb = estimator_xgb_여름.predict(X_test_여름)
get_clf_eval(y_test_여름, y_best_pred_여름_xgb)

오차행렬:
 [[241  20]
 [ 28 128]]

정확도: 0.8849
정밀도: 0.8649
재현율: 0.8205
F1: 0.8421
AUC: 0.8719


In [133]:
# 재현율이 가장 높은 XGB모델 저장
import joblib

joblib.dump(estimator_xgb_여름 ,"/content/gdrive/MyDrive/프로젝트_모델/xgb_여름_최종.pkl")

['/content/gdrive/MyDrive/프로젝트_모델/xgb_여름_최종.pkl']

# 가을

In [102]:
data_가을 = pd.read_csv("/content/gdrive/MyDrive/산단공_프로젝트/가을_중업종재해반영x.csv", encoding = "cp949")

In [103]:
id_가을 = pd.Series(data_가을.id, name = 'id') # 그리드 id만 추출해서 저장
data_가을.drop(['Unnamed: 0', 'Unnamed: 0.1', "단지명", "id"], axis = 1, inplace = True)

In [104]:
# 중대재해가 1번 이상 발생했으면 1, 발생하지 않았으면 0

data_가을.loc[data_가을['중대재해유무'] == True, "중대재해유무"] = 1
data_가을.loc[data_가을['중대재해유무'] == False, "중대재해유무"] = 0

# 필요없는 컬럼 제거
data_가을.drop(['발생형태_끼임', '발생형태_작업관련질병(뇌심등)',
       '발생형태_직업병(진폐제외)', '발생형태_넘어짐', '발생형태_물체에맞음', '발생형태_부딪힘', '발생형태_떨어짐',
       '발생형태_절단·베임·찔림', '발생형태_불균형및무리한동작', '발생형태_이상온도접촉', '발생형태_깔림·뒤집힘',
       '발생형태_사업장외교통사고', '발생형태_체육행사등의사고', '발생형태_화학물질누출·접촉', '발생형태_폭발·파열',
       '발생형태_화재', '발생형태_감전', '발생형태_진폐', '발생형태_폭력행위', '발생형태_무너짐',
       '발생형태_사업장내교통사고', '발생형태_기타', '발생형태_분류불능', '발생형태_산소결핍', '발생형태_빠짐ㆍ익사',
       '발생형태_동물상해', '근속기간_10년이상', '근속기간_1년미만', '근속기간_5년~10년미만', '근속기간_1년~2년미만',
       '근속기간_2년~3년미만', '근속기간_3년~4년미만', '근속기간_4년~5년미만', '연령_50세~59세',
       '연령_60세~69세', '연령_40세~49세', '연령_30세~39세', '연령_20세~29세', '연령_70세이상', ], axis = 1, inplace = True)

In [105]:
# 관리기관 원-핫 인코딩
data_가을 = pd.get_dummies(data = data_가을, columns = ['관리기관'], prefix = '관리기관')

In [106]:
# 변수 스케일링

from sklearn.preprocessing import MinMaxScaler
scaler_ = MinMaxScaler()
data_가을_scaled_ = scaler_.fit_transform(data_가을)

data_가을 = pd.DataFrame(data_가을_scaled_, columns = data_가을.columns)

In [107]:
# 중대재해유무: target

X_가을 = data_가을.drop(["중대재해유무"], axis = 1)
y_가을 = data_가을.중대재해유무

In [108]:
# 학습 데이터와 테스트 데이터 나눔
# 중대재해발생유무 비율에 맞도록 분리

X_train_가을, X_test_가을, y_train_가을, y_test_가을 = train_test_split(X_가을, y_가을, random_state = 42, stratify=y_가을)

## RandomForest

In [109]:
rf_clf_가을 = RandomForestClassifier()
rf_clf_가을.fit(X_train_가을, y_train_가을)

y_rf_pred_가을 = rf_clf_가을.predict(X_test_가을)

get_clf_eval(y_test_가을, y_rf_pred_가을)

오차행렬:
 [[313   5]
 [ 24  49]]

정확도: 0.9258
정밀도: 0.9074
재현율: 0.6712
F1: 0.7717
AUC: 0.8278


In [110]:
# RF 모델 하이퍼 파라미터 튜닝

rf_가을_best = RandomForestClassifier()

# 파라미터 지정
rf_가을_parameters ={ 'n_estimators' : [100, 200, 300],
           'max_depth' : [6, 8, 10, 12],
           'min_samples_leaf' : [1, 3, 5, 8],
           'min_samples_split' : [1, 3, 5, 8]
            }
rf_grid_가을=GridSearchCV(rf_가을_best, param_grid = rf_가을_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


rf_grid_가을.fit(X_train_가을, y_train_가을)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(rf_grid_가을.best_score_))
print("best param : ",rf_grid_가을.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(rf_grid_가을.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.7419
best param :  {'max_depth': 12, 'min_samples_leaf': 3, 'min_samples_split': 3, 'n_estimators': 100}


,params,mean_test_score,rank_test_score
159,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.741882,1
125,"{'max_depth': 10, 'min_samples_leaf': 5, 'min_...",0.737253,2
153,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.737253,2
154,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.737253,2
155,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.737253,2
160,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.737253,2
163,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.737253,2
164,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.737253,2
116,"{'max_depth': 10, 'min_samples_leaf': 3, 'min_...",0.737253,2
117,"{'max_depth': 10, 'min_samples_leaf': 3, 'min_...",0.737253,2


In [111]:
estimator_rf_가을 = rf_grid_가을.best_estimator_
y_best_pred_가을_rf = estimator_rf_가을.predict(X_test_가을)
get_clf_eval(y_test_가을, y_best_pred_가을_rf)

오차행렬:
 [[313   5]
 [ 23  50]]

정확도: 0.9284
정밀도: 0.9091
재현율: 0.6849
F1: 0.7812
AUC: 0.8346


## 서포트 벡터

In [112]:
svc_가을 = SVC(kernel='poly').fit(X_train_가을, y_train_가을)
svc_가을.fit(X_train_가을, y_train_가을)

y_svc_pred_가을 = svc_가을.predict(X_test_가을)

get_clf_eval(y_test_가을, y_svc_pred_가을)

오차행렬:
 [[317   1]
 [ 67   6]]

정확도: 0.8261
정밀도: 0.8571
재현율: 0.0822
F1: 0.1500
AUC: 0.5395


## XGB

In [113]:
xgb_가을 = XGBClassifier(early_sropping_rounds = 50, random_state = 42)
xgb_가을.fit(X_train_가을, y_train_가을)

y_xgb_pred_가을 = xgb_가을.predict(X_test_가을)

get_clf_eval(y_test_가을, y_xgb_pred_가을)

오차행렬:
 [[309   9]
 [ 23  50]]

정확도: 0.9182
정밀도: 0.8475
재현율: 0.6849
F1: 0.7576
AUC: 0.8283


In [114]:
# XGB 모델 하이퍼 파라미터 튜닝

xgb_가을_best = XGBClassifier(early_sropping_rounds = 50, random_state = 42)

# 파라미터 지정
xg_가을_parameters ={'n_estimators' : [100,200,300],
    'learning_rate' : [0.01,0.05,0.1,0.15],
    'max_depth' : [3,5,7,10],
    'gamma' : [0,1,2,3],
    'colsample_bytree' : [0.8,0.9]}
xgb_grid_가을=GridSearchCV(xgb_가을_best, param_grid = xg_가을_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


xgb_grid_가을.fit(X_train_가을, y_train_가을)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(xgb_grid_가을.best_score_))
print("best param : ",xgb_grid_가을.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(xgb_grid_가을.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.7418
best param :  {'colsample_bytree': 0.9, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100}


,params,mean_test_score,rank_test_score
198,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.741755,1
195,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.741755,1
291,"{'colsample_bytree': 0.9, 'gamma': 2, 'learnin...",0.737253,3
241,"{'colsample_bytree': 0.9, 'gamma': 1, 'learnin...",0.737253,3
373,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.737253,3
372,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.737253,3
338,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.737253,3
297,"{'colsample_bytree': 0.9, 'gamma': 2, 'learnin...",0.737253,3
193,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.737253,3
201,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.737126,10


In [115]:
estimator_xgb_가을 = xgb_grid_가을.best_estimator_
y_best_pred_가을_xgb = estimator_xgb_가을.predict(X_test_가을)
get_clf_eval(y_test_가을, y_best_pred_가을_xgb)

오차행렬:
 [[306  12]
 [ 23  50]]

정확도: 0.9105
정밀도: 0.8065
재현율: 0.6849
F1: 0.7407
AUC: 0.8236


In [134]:
# 재현율이 가장 높은 RF모델 저장
import joblib

joblib.dump(estimator_rf_가을 ,"/content/gdrive/MyDrive/프로젝트_모델/rf_가을_최종.pkl")

['/content/gdrive/MyDrive/프로젝트_모델/rf_가을_최종.pkl']

# 겨울

In [117]:
data_겨울 = pd.read_csv("/content/gdrive/MyDrive/산단공_프로젝트/겨울_중업종재해반영x.csv", encoding = "cp949")

In [118]:
id_겨울 = pd.Series(data_겨울.id, name = 'id') # 그리드 id만 추출해서 저장
data_겨울.drop(['Unnamed: 0', 'Unnamed: 0.1', "단지명", "id"], axis = 1, inplace = True)

In [119]:
# 중대재해가 1번 이상 발생했으면 1, 발생하지 않았으면 0

data_겨울.loc[data_겨울['중대재해유무'] == True, "중대재해유무"] = 1
data_겨울.loc[data_겨울['중대재해유무'] == False, "중대재해유무"] = 0

# 필요없는 컬럼 제거
data_겨울.drop(['발생형태_끼임', '발생형태_작업관련질병(뇌심등)',
       '발생형태_직업병(진폐제외)', '발생형태_넘어짐', '발생형태_물체에맞음', '발생형태_부딪힘', '발생형태_떨어짐',
       '발생형태_절단·베임·찔림', '발생형태_불균형및무리한동작', '발생형태_이상온도접촉', '발생형태_깔림·뒤집힘',
       '발생형태_사업장외교통사고', '발생형태_체육행사등의사고', '발생형태_화학물질누출·접촉', '발생형태_폭발·파열',
       '발생형태_화재', '발생형태_감전', '발생형태_진폐', '발생형태_폭력행위', '발생형태_무너짐',
       '발생형태_사업장내교통사고', '발생형태_기타', '발생형태_분류불능', '발생형태_산소결핍', '발생형태_빠짐ㆍ익사',
       '발생형태_동물상해', '근속기간_10년이상', '근속기간_1년미만', '근속기간_5년~10년미만', '근속기간_1년~2년미만',
       '근속기간_2년~3년미만', '근속기간_3년~4년미만', '근속기간_4년~5년미만', '연령_50세~59세',
       '연령_60세~69세', '연령_40세~49세', '연령_30세~39세', '연령_20세~29세', '연령_70세이상', ], axis = 1, inplace = True)

In [120]:
# 관리기관 원-핫 인코딩
data_겨울 = pd.get_dummies(data = data_겨울, columns = ['관리기관'], prefix = '관리기관')

In [121]:
# 변수 스케일링

from sklearn.preprocessing import MinMaxScaler
scaler_ = MinMaxScaler()
data_겨울_scaled_ = scaler_.fit_transform(data_겨울)

data_겨울 = pd.DataFrame(data_겨울_scaled_, columns = data_겨울.columns)

In [122]:
# 중대재해유무: target

X_겨울 = data_겨울.drop(["중대재해유무"], axis = 1)
y_겨울 = data_겨울.중대재해유무

In [123]:
# 학습 데이터와 테스트 데이터 나눔
# 중대재해발생유무 비율에 맞도록 분리

X_train_겨울, X_test_겨울, y_train_겨울, y_test_겨울 = train_test_split(X_겨울, y_겨울, random_state = 42, stratify=y_겨울)

## RandomForest

In [124]:
rf_clf_겨울 = RandomForestClassifier()
rf_clf_겨울.fit(X_train_겨울, y_train_겨울)

y_rf_pred_겨울 = rf_clf_겨울.predict(X_test_겨울)

get_clf_eval(y_test_겨울, y_rf_pred_겨울)

오차행렬:
 [[187  33]
 [ 24 132]]

정확도: 0.8484
정밀도: 0.8000
재현율: 0.8462
F1: 0.8224
AUC: 0.8481


In [125]:
# RF 모델 하이퍼 파라미터 튜닝

rf_겨울_best = RandomForestClassifier()

# 파라미터 지정
rf_겨울_parameters ={ 'n_estimators' : [100, 200, 300],
           'max_depth' : [6, 8, 10, 12],
           'min_samples_leaf' : [1, 3, 5, 8],
           'min_samples_split' : [1, 3, 5, 8]
            }
rf_grid_겨울=GridSearchCV(rf_겨울_best, param_grid = rf_겨울_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


rf_grid_겨울.fit(X_train_겨울, y_train_겨울)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(rf_grid_겨울.best_score_))
print("best param : ",rf_grid_겨울.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(rf_grid_겨울.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.7957
best param :  {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 5, 'n_estimators': 100}


,params,mean_test_score,rank_test_score
126,"{'max_depth': 10, 'min_samples_leaf': 5, 'min_...",0.795699,1
166,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.795699,1
187,"{'max_depth': 12, 'min_samples_leaf': 8, 'min_...",0.795699,3
105,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_...",0.793548,4
173,"{'max_depth': 12, 'min_samples_leaf': 5, 'min_...",0.793548,4
127,"{'max_depth': 10, 'min_samples_leaf': 5, 'min_...",0.793548,4
163,"{'max_depth': 12, 'min_samples_leaf': 3, 'min_...",0.791398,7
147,"{'max_depth': 12, 'min_samples_leaf': 1, 'min_...",0.791398,7
111,"{'max_depth': 10, 'min_samples_leaf': 3, 'min_...",0.791398,7
176,"{'max_depth': 12, 'min_samples_leaf': 5, 'min_...",0.789247,10


In [126]:
estimator_rf_겨울 = rf_grid_겨울.best_estimator_
y_best_pred_겨울_rf = estimator_rf_겨울.predict(X_test_겨울)
get_clf_eval(y_test_겨울, y_best_pred_겨울_rf)

오차행렬:
 [[190  30]
 [ 25 131]]

정확도: 0.8537
정밀도: 0.8137
재현율: 0.8397
F1: 0.8265
AUC: 0.8517


## 서포트 벡터

In [127]:
svc_겨울 = SVC(kernel='poly').fit(X_train_겨울, y_train_겨울)
svc_겨울.fit(X_train_겨울, y_train_겨울)

y_svc_pred_겨울 = svc_겨울.predict(X_test_겨울)

get_clf_eval(y_test_겨울, y_svc_pred_겨울)

오차행렬:
 [[198  22]
 [113  43]]

정확도: 0.6410
정밀도: 0.6615
재현율: 0.2756
F1: 0.3891
AUC: 0.5878


## XGB

In [128]:
xgb_겨울 = XGBClassifier(early_sropping_rounds = 50, random_state = 42)
xgb_겨울.fit(X_train_겨울, y_train_겨울)

y_xgb_pred_겨울 = xgb_겨울.predict(X_test_겨울)

get_clf_eval(y_test_겨울, y_xgb_pred_겨울)

오차행렬:
 [[188  32]
 [ 26 130]]

정확도: 0.8457
정밀도: 0.8025
재현율: 0.8333
F1: 0.8176
AUC: 0.8439


In [129]:
# XGB 모델 하이퍼 파라미터 튜닝

xgb_겨울_best = XGBClassifier(early_sropping_rounds = 50, random_state = 42)

# 파라미터 지정
xg_겨울_parameters ={'n_estimators' : [100,200,300],
    'learning_rate' : [0.01,0.05,0.1,0.15],
    'max_depth' : [3,5,7,10],
    'gamma' : [0,1,2,3],
    'colsample_bytree' : [0.8,0.9]}
xgb_grid_겨울=GridSearchCV(xgb_겨울_best, param_grid = xg_겨울_parameters, scoring="recall", cv = 3) # scoring을 재현율로 지정


xgb_grid_겨울.fit(X_train_겨울, y_train_겨울)

#best f1_macro 수치와 best parameter확인
print("best f1_macro : {0: .4f}".format(xgb_grid_겨울.best_score_))
print("best param : ",xgb_grid_겨울.best_params_)

#dataframe으로 랭킹순보기
result_df = pd.DataFrame(xgb_grid_겨울.cv_results_)
result_df.sort_values(by=['rank_test_score'],inplace=True)

#plot
result_df[['params','mean_test_score','rank_test_score']].head(10)


best f1_macro :  0.8086
best param :  {'colsample_bytree': 0.9, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}


,params,mean_test_score,rank_test_score
240,"{'colsample_bytree': 0.9, 'gamma': 1, 'learnin...",0.808602,1
288,"{'colsample_bytree': 0.9, 'gamma': 2, 'learnin...",0.808602,1
192,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.808602,1
241,"{'colsample_bytree': 0.9, 'gamma': 1, 'learnin...",0.804301,4
193,"{'colsample_bytree': 0.9, 'gamma': 0, 'learnin...",0.804301,4
49,"{'colsample_bytree': 0.8, 'gamma': 1, 'learnin...",0.802151,6
289,"{'colsample_bytree': 0.9, 'gamma': 2, 'learnin...",0.802151,6
97,"{'colsample_bytree': 0.8, 'gamma': 2, 'learnin...",0.797849,8
1,"{'colsample_bytree': 0.8, 'gamma': 0, 'learnin...",0.797849,8
336,"{'colsample_bytree': 0.9, 'gamma': 3, 'learnin...",0.795699,10


In [130]:
estimator_xgb_겨울 = xgb_grid_겨울.best_estimator_
y_best_pred_겨울_xgb = estimator_xgb_겨울.predict(X_test_겨울)
get_clf_eval(y_test_겨울, y_best_pred_겨울_xgb)

오차행렬:
 [[180  40]
 [ 24 132]]

정확도: 0.8298
정밀도: 0.7674
재현율: 0.8462
F1: 0.8049
AUC: 0.8322


In [135]:
# 재현율이 가장 높은 XGB모델 저장
import joblib

joblib.dump(estimator_xgb_겨울 ,"/content/gdrive/MyDrive/프로젝트_모델/xgb_겨울_최종.pkl")

['/content/gdrive/MyDrive/프로젝트_모델/xgb_겨울_최종.pkl']